In [1]:
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("CYTB_results/CYTB_all_candidates.csv")
OUTPUT_FILE = Path("CYTB_results/CYTB_selected_sequences.csv")

df = pd.read_csv(INPUT_FILE, sep=None, engine="python")

df["Partial"] = (
    df["Partial"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

df["Length_difference"] = abs(df["Length_aa"] - 380)

# Предпочитаем полные последовательности
df = df.sort_values(
    by=["Partial", "Length_difference", "Score"],
    ascending=[True, True, False]
)

# Определяем столбец с видом
species_column = "Species" if "Species" in df.columns else "NCBI_organism"

# Выбираем одну последовательность на вид
selected = df.drop_duplicates(subset=species_column, keep="first")

selected.to_csv(OUTPUT_FILE, index=False)

print(f"Всего кандидатов: {len(df)}")
print(f"Выбрано последовательностей: {len(selected)}")
print(f"Видов: {selected[species_column].nunique()}")
print(f"Файл сохранён: {OUTPUT_FILE}")

print("\nВыбранные последовательности:")
print(selected[
    [species_column, "Accession", "Length_aa", "Partial", "Score"]
].to_string(index=False))

KeyError: 'Score'

In [9]:
import pandas as pd
from Bio import SeqIO
from pathlib import Path

BASE_DIR = Path("CYTB_results")

CSV_FILE = BASE_DIR / "CYTB_selected_sequences.csv"
FASTA_FILE = BASE_DIR / "CYTB_all_sequences.fasta"
OUTPUT_FASTA = BASE_DIR / "CYTB_selected_sequences.fasta"
MISSING_FILE = BASE_DIR / "CYTB_selected_missing_accessions.txt"

df = pd.read_csv(CSV_FILE, encoding="utf-8-sig")

selected = {
    str(row["Accession"]).strip(): str(row["Host_species"]).strip()
    for _, row in df.iterrows()
}

selected_records = {}
for record in SeqIO.parse(FASTA_FILE, "fasta"):
    if record.id in selected:
        selected_records[record.id] = record

output_records = []
for accession, species in selected.items():
    if accession in selected_records:
        record = selected_records[accession]
        record.id = species.replace(" ", "_")
        record.name = record.id
        record.description = ""
        output_records.append(record)

SeqIO.write(output_records, OUTPUT_FASTA, "fasta")

missing = set(selected) - set(selected_records)

with open(MISSING_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(missing)))

print(f"Выбрано в CSV: {len(selected)}")
print(f"Найдено в FASTA: {len(output_records)}")
print(f"Не найдено: {len(missing)}")
print(f"FASTA: {OUTPUT_FASTA}")

Выбрано в CSV: 28
Найдено в FASTA: 28
Не найдено: 0
FASTA: CYTB_results/CYTB_selected_sequences.fasta


In [7]:
from Bio import SeqIO

for record in SeqIO.parse("CYTB_results/CYTB_all_sequences.fasta", "fasta"):
    print(record.description)
    if record:
        break

YHG28083.1 cytochrome b (mitochondrion) [Gallus gallus]


In [3]:
from pathlib import Path

import pandas as pd

source_csv = Path(
    "/Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/"
    "snakemake/constants/Astroviridae_Aves_Amphibia_14072026_host_filtered.csv"
)
mapping_csv = Path(
    "/Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/scripts/CYTB_results/CYTB_host_mapping.csv"
)

for path in (source_csv, mapping_csv):
    if not path.is_file():
        raise FileNotFoundError(f"Файл не найден — {path}")

data = pd.read_csv(source_csv, dtype=str, keep_default_na=False, encoding="utf-8-sig")
mapping = pd.read_csv(mapping_csv, dtype=str, keep_default_na=False, encoding="utf-8-sig")

allowed_hosts = set(mapping["host_from_genbank"].str.strip())
allowed_hosts.discard("")

good_host = data["Host_species"].str.strip().isin(allowed_hosts)
data["is_good_host"] = good_host.astype(int)

good_ids = data.loc[good_host, "ID"].str.strip()
good_ids = good_ids[good_ids.ne("")].drop_duplicates()

output_dir = mapping_csv.parent
output_csv = output_dir / source_csv.name
output_txt = output_dir / "good_host_ids.txt"

data.to_csv(output_csv, index=False, encoding="utf-8")
output_txt.write_text(
    "".join(f"{identifier}\n" for identifier in good_ids),
    encoding="utf-8",
)

print(f"Строк с is_good_host = 1 — {good_host.sum()} из {len(data)}")
print(f"Уникальных ID — {len(good_ids)}")
print(f"CSV — {output_csv}")
print(f"TXT — {output_txt}")

Строк с is_good_host = 1 — 368 из 396
Уникальных ID — 368
CSV — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/scripts/CYTB_results/Astroviridae_Aves_Amphibia_14072026_host_filtered.csv
TXT — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/scripts/CYTB_results/good_host_ids.txt


In [4]:
from pathlib import Path

data_dir = Path.home() / "Desktop/lab/Astroviridae_database/data"
ids_file = data_dir / "host_analysis/CYTB_results/good_host_ids_cytb.txt"
fasta_dir = data_dir / "clade_analysis/AVES/final_files"

ids = {
    line.strip().rstrip("\\").strip()
    for line in ids_file.read_text(encoding="utf-8-sig").splitlines()
    if line.strip()
}
ids.discard("")

if not ids:
    raise ValueError(f"Список ID пуст — {ids_file}")

for region in ("1A", "1B", "2"):
    input_fasta = fasta_dir / f"Aves_full_seq_removed_{region}.fasta"
    output_fasta = ids_file.parent / f"Aves_full_seq_removed_{region}_cytb_hosts.fasta"

    found_ids = set()
    keep_record = False

    with (
        input_fasta.open(encoding="utf-8", newline="") as source,
        output_fasta.open("w", encoding="utf-8", newline="") as output,
    ):
        for line in source:
            if line.startswith(">"):
                # Заголовок имеет вид >PX237222/NA/...
                accession = line[1:].split("/", 1)[0].split()[0]
                keep_record = accession in ids
                if keep_record:
                    found_ids.add(accession)

            if keep_record:
                output.write(line)

    missing_ids = ids - found_ids
    print(f"{region} — найдено {len(found_ids)} из {len(ids)} ID")
    print(f"Файл — {output_fasta}")
    if missing_ids:
        print("Отсутствуют —", ", ".join(sorted(missing_ids)))

1A — найдено 368 из 368 ID
Файл — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/host_analysis/CYTB_results/Aves_full_seq_removed_1A_cytb_hosts.fasta
1B — найдено 368 из 368 ID
Файл — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/host_analysis/CYTB_results/Aves_full_seq_removed_1B_cytb_hosts.fasta
2 — найдено 368 из 368 ID
Файл — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/host_analysis/CYTB_results/Aves_full_seq_removed_2_cytb_hosts.fasta


In [1]:
from pathlib import Path

data_dir = Path.home() / "Desktop/lab/Astroviridae_database/data"
ids_file = data_dir / "host_analysis/OpenTree_results/good_host_ids_otol.txt"
fasta_dir = data_dir / "clade_analysis/AVES/final_files"

ids = {
    line.strip().rstrip("\\").strip()
    for line in ids_file.read_text(encoding="utf-8-sig").splitlines()
    if line.strip()
}
ids.discard("")

if not ids:
    raise ValueError(f"Список ID пуст — {ids_file}")

for region in ("1A", "1B", "2"):
    input_fasta = fasta_dir / f"Aves_full_seq_removed_{region}.fasta"
    output_fasta = ids_file.parent / f"Aves_full_seq_removed_{region}_cytb_hosts.fasta"

    found_ids = set()
    keep_record = False

    with (
        input_fasta.open(encoding="utf-8", newline="") as source,
        output_fasta.open("w", encoding="utf-8", newline="") as output,
    ):
        for line in source:
            if line.startswith(">"):
                # Заголовок имеет вид >PX237222/NA/...
                accession = line[1:].split("/", 1)[0].split()[0]
                keep_record = accession in ids
                if keep_record:
                    found_ids.add(accession)

            if keep_record:
                output.write(line)

    missing_ids = ids - found_ids
    print(f"{region} — найдено {len(found_ids)} из {len(ids)} ID")
    print(f"Файл — {output_fasta}")
    if missing_ids:
        print("Отсутствуют —", ", ".join(sorted(missing_ids)))

1A — найдено 371 из 371 ID
Файл — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/host_analysis/OpenTree_results/Aves_full_seq_removed_1A_cytb_hosts.fasta
1B — найдено 371 из 371 ID
Файл — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/host_analysis/OpenTree_results/Aves_full_seq_removed_1B_cytb_hosts.fasta
2 — найдено 371 из 371 ID
Файл — /Users/abagavetdinova/Desktop/lab/Astroviridae_database/data/host_analysis/OpenTree_results/Aves_full_seq_removed_2_cytb_hosts.fasta
